# 4.2 Resting State Classification Performance (H1)

**Hypothesis H1:** A one-vs-rest logistic regression classifier trained on full-brain functional connectivity patterns significantly exceeds chance-level performance in identifying brain regions during resting state, and generalises to unseen task data.

**Files used:**
- `data/results/full_connectivity_analysis/one_vs_rest/cv_summary.json`
- `data/results/full_connectivity_analysis/one_vs_rest/overall_metrics.json`
- `data/results/full_connectivity_analysis/one_vs_rest/confusion_matrix.npy`
- `data/results/full_connectivity_analysis/one_vs_rest/cv_true_labels.npy`
- `data/results/full_connectivity_analysis/one_vs_rest/cv_predictions.npy`
- `data/results/full_connectivity_analysis/task_testing_one_vs_rest/task_testing_summary.json`
- `data/results/full_connectivity_analysis/task_testing_one_vs_rest/task_true_labels.npy`
- `data/results/full_connectivity_analysis/task_testing_one_vs_rest/task_predictions.npy`

## Cell 1 — Imports and Path Setup

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths — notebook lives at analysis/hypothesis_testing/
# so project root is two levels up
# ---------------------------------------------------------------------------
ROOT = Path("../..")  # brain_connectivity_classifier/

OVR_DIR   = ROOT / "data/results/full_connectivity_analysis/one_vs_rest"
TASK_DIR  = ROOT / "data/results/full_connectivity_analysis/task_testing_one_vs_rest"

# Verify all required files exist before proceeding
required_files = [
    OVR_DIR  / "cv_summary.json",
    OVR_DIR  / "overall_metrics.json",
    OVR_DIR  / "confusion_matrix.npy",
    OVR_DIR  / "cv_true_labels.npy",
    OVR_DIR  / "cv_predictions.npy",
    TASK_DIR / "task_testing_summary.json",
    TASK_DIR / "task_true_labels.npy",
    TASK_DIR / "task_predictions.npy",
]

for f in required_files:
    status = "✓" if f.exists() else "✗ MISSING"
    print(f"{status}  {f.relative_to(ROOT)}")

✓  data/results/full_connectivity_analysis/one_vs_rest/cv_summary.json
✓  data/results/full_connectivity_analysis/one_vs_rest/overall_metrics.json
✓  data/results/full_connectivity_analysis/one_vs_rest/confusion_matrix.npy
✓  data/results/full_connectivity_analysis/one_vs_rest/cv_true_labels.npy
✓  data/results/full_connectivity_analysis/one_vs_rest/cv_predictions.npy
✓  data/results/full_connectivity_analysis/task_testing_one_vs_rest/task_testing_summary.json
✓  data/results/full_connectivity_analysis/task_testing_one_vs_rest/task_true_labels.npy
✓  data/results/full_connectivity_analysis/task_testing_one_vs_rest/task_predictions.npy


## Cell 2 — Load Core Summary Files

In [ ]:
with open(OVR_DIR / "cv_summary.json")           as f: cv_summary    = json.load(f)
with open(OVR_DIR / "overall_metrics.json")       as f: overall       = json.load(f)
with open(TASK_DIR / "task_testing_summary.json") as f: task_summary  = json.load(f)

cv_true_labels   = np.load(OVR_DIR  / "cv_true_labels.npy")
cv_predictions   = np.load(OVR_DIR  / "cv_predictions.npy")
confusion_matrix = np.load(OVR_DIR  / "confusion_matrix.npy")
task_true_labels = np.load(TASK_DIR / "task_true_labels.npy")
task_predictions = np.load(TASK_DIR / "task_predictions.npy")

print("cv_summary keys     :", list(cv_summary.keys()))
print("overall_metrics keys:", list(overall.keys()))
print("task_summary keys   :", list(task_summary.keys()))

cv_summary keys     : ['diagonal_strategy', 'best_fold_hyperparameters', 'best_fold_idx', 'best_fold_val_accuracy', 'overall_metrics', 'fold_metrics', 'mean_train_accuracy', 'mean_val_accuracy', 'generalization_gap', 'mean_task_accuracy', 'mean_rest_task_drop', 'cv_time_seconds', 'paired_ttest_note', 'output_files', 'cross_state_analysis']
overall_metrics keys: ['accuracy', 'n_samples', 'n_classes', 'top_3_accuracy', 'top_5_accuracy', 'top_10_accuracy']
task_summary keys   : ['note', 'diagonal_strategy', 'rest_train_accuracy', 'task_test_accuracy', 'accuracy_drop', 'hyperparameters', 'n_rest_subjects', 'n_task_subjects', 'n_regions', 'training_time_seconds', 'rest_top_3_accuracy', 'task_top_3_accuracy', 'rest_top_5_accuracy', 'task_top_5_accuracy', 'rest_top_10_accuracy', 'task_top_10_accuracy']


## Cell 3 — Extract Top-Level Accuracies and Dataset Constants

In [ ]:
N_CLASSES = 232
CHANCE    = 1 / N_CLASSES  # ≈ 0.0043

# Resting-state CV accuracy
rest_cv_acc = (
    cv_summary.get("mean_val_accuracy")
    or overall.get("accuracy")
)

# Task generalisation accuracy
task_acc = (
    cv_summary.get("mean_task_accuracy")
    or task_summary.get("task_test_accuracy")
    or task_summary.get("accuracy")
)

# Task set size
n_task_samples = len(task_true_labels)

print(f"N classes         : {N_CLASSES}")
print(f"Chance level      : {CHANCE:.4f} ({CHANCE*100:.2f}%)")
print(f"Rest CV accuracy  : {rest_cv_acc:.4f} ({rest_cv_acc*100:.2f}%)")
print(f"Task accuracy     : {task_acc:.4f} ({task_acc*100:.2f}%)")
print(f"Task samples (n)  : {n_task_samples:,}")

N classes         : 232
Chance level      : 0.0043 (0.43%)
Rest CV accuracy  : 0.7624 (76.24%)
Task accuracy     : 0.7285 (72.85%)
Task samples (n)  : 46,400


## Cell 4 — Compute Table 1 Metrics

In [ ]:
abs_drop  = rest_cv_acc - task_acc
rel_drop  = (abs_drop / rest_cv_acc) * 100
error_count = int(np.sum(task_predictions != task_true_labels))
error_rate  = error_count / n_task_samples

# Cross-check: accuracy from confusion matrix diagonal
cm_acc = confusion_matrix.diagonal().sum() / confusion_matrix.sum()

# Cross-check: accuracy recomputed from raw arrays
cv_acc_recomputed = np.mean(cv_predictions == cv_true_labels)

print(f"Absolute drop          : {abs_drop*100:.2f}%")
print(f"Relative drop          : {rel_drop:.2f}%")
print(f"Task error count       : {error_count:,}")
print(f"Task error rate        : {error_rate:.4f} ({error_rate*100:.2f}%)")
print()
print("--- Cross-checks ---")
print(f"CM diagonal accuracy   : {cm_acc*100:.2f}%  (should match Rest CV)")
print(f"Raw array CV accuracy  : {cv_acc_recomputed*100:.2f}%  (should match Rest CV)")

Absolute drop          : 3.40%
Relative drop          : 4.45%
Task error count       : 12,511
Task error rate        : 0.2696 (26.96%)

--- Cross-checks ---
CM diagonal accuracy   : 76.24%  (should match Rest CV)
Raw array CV accuracy  : 76.24%  (should match Rest CV)


## Cell 5 — Build and Display Table 1

In [ ]:
table1 = pd.DataFrame([
    {
        "Metric"            : "Resting-State CV Accuracy",
        "Value"             : f"{rest_cv_acc*100:.2f}%",
        "Notes"             : "Mean across 5 folds"
    },
    {
        "Metric"            : "Task Generalisation Accuracy",
        "Value"             : f"{task_acc*100:.2f}%",
        "Notes"             : "Held-out task set"
    },
    {
        "Metric"            : "Absolute Drop (Rest → Task)",
        "Value"             : f"{abs_drop*100:.2f}%",
        "Notes"             : "Rest CV − Task accuracy"
    },
    {
        "Metric"            : "Relative Drop",
        "Value"             : f"{rel_drop:.2f}%",
        "Notes"             : "Absolute drop / Rest CV × 100"
    },
    {
        "Metric"            : "Task Error Count",
        "Value"             : f"{error_count:,}",
        "Notes"             : f"Out of {n_task_samples:,} samples"
    },
    {
        "Metric"            : "Task Error Rate",
        "Value"             : f"{error_rate*100:.2f}%",
        "Notes"             : "Error count / n_task_samples"
    },
    {
        "Metric"            : "Chance Level",
        "Value"             : f"{CHANCE*100:.2f}%",
        "Notes"             : f"1 / {N_CLASSES} classes"
    },
    {
        "Metric"            : "Performance vs. Chance",
        "Value"             : f"{rest_cv_acc / CHANCE:.0f}×",
        "Notes"             : "Rest CV / chance level"
    },
])

table1.set_index("Metric", inplace=True)
table1

,Value,Notes
Metric,,
Resting-State CV Accuracy,76.24%,Mean across 5 folds
Task Generalisation Accuracy,72.85%,Held-out task set
Absolute Drop (Rest → Task),3.40%,Rest CV − Task accuracy
Relative Drop,4.45%,Absolute drop / Rest CV × 100
Task Error Count,"12,511","Out of 46,400 samples"
Task Error Rate,26.96%,Error count / n_task_samples
Chance Level,0.43%,1 / 232 classes
Performance vs. Chance,177×,Rest CV / chance level


## Cell 6 — Extract Per-Fold Accuracies for Statistical Tests

In [ ]:
fold_metrics = cv_summary["fold_metrics"]  # list of 5 dicts

rest_folds = np.array([fm["val_accuracy"]  for fm in fold_metrics])
task_folds = np.array([fm["task_accuracy"] for fm in fold_metrics])

fold_df = pd.DataFrame({
    "Fold"           : [f"Fold {i+1}" for i in range(len(fold_metrics))],
    "Rest CV Acc"    : rest_folds,
    "Task Acc"       : task_folds,
    "Fold Drop"      : rest_folds - task_folds,
})

fold_df.set_index("Fold", inplace=True)
fold_df

,Rest CV Acc,Task Acc,Fold Drop
Fold,,,
Fold 1,0.762261,0.731228,0.031032
Fold 2,0.761973,0.729741,0.032232
Fold 3,0.751628,0.722931,0.028697
Fold 4,0.774042,0.728103,0.045939
Fold 5,0.762245,0.730323,0.031922


## Cell 7 — Statistical Test 1: One-Sample t-Test (CV Accuracy vs. Chance)

In [ ]:
t1, p1 = stats.ttest_1samp(rest_folds, popmean=CHANCE)

# Cohen's d = (mean - mu) / std
d1 = (rest_folds.mean() - CHANCE) / rest_folds.std(ddof=1)

print("One-sample t-test: Rest CV accuracy vs. chance level")
print(f"  Mean rest CV   : {rest_folds.mean()*100:.2f}%")
print(f"  Chance level   : {CHANCE*100:.2f}%")
print(f"  t({len(rest_folds)-1})           : {t1:.1f}")
print(f"  p-value        : {p1:.2e}")
print(f"  Cohen's d      : {d1:.1f}")

One-sample t-test: Rest CV accuracy vs. chance level
  Mean rest CV   : 76.24%
  Chance level   : 0.43%
  t(4)           : 213.7
  p-value        : 2.88e-09
  Cohen's d      : 95.6


## Cell 8 — Statistical Test 2: Paired t-Test (Rest CV vs. Task Accuracy Across Folds)

In [ ]:
t2, p2 = stats.ttest_rel(rest_folds, task_folds)

# Cohen's d for paired test = mean difference / std of differences
diffs = rest_folds - task_folds
d2 = diffs.mean() / diffs.std(ddof=1)

print("Paired t-test: Rest CV accuracy vs. Task accuracy (across 5 folds)")
print(f"  Mean rest CV   : {rest_folds.mean()*100:.2f}%")
print(f"  Mean task acc  : {task_folds.mean()*100:.2f}%")
print(f"  Mean diff      : {diffs.mean()*100:.2f}%")
print(f"  t({len(rest_folds)-1})           : {t2:.1f}")
print(f"  p-value        : {p2:.3f}")
print(f"  Cohen's d      : {d2:.2f}")

Paired t-test: Rest CV accuracy vs. Task accuracy (across 5 folds)
  Mean rest CV   : 76.24%
  Mean task acc  : 72.85%
  Mean diff      : 3.40%
  t(4)           : 11.1
  p-value        : 0.000
  Cohen's d      : 4.97


## Cell 9 — Statistical Results Summary Table

In [ ]:
stats_table = pd.DataFrame([
    {
        "Test"       : "One-sample t-test (CV vs. chance)",
        "t-statistic": f"t({len(rest_folds)-1}) = {t1:.1f}",
        "p-value"    : f"{p1:.2e}",
        "Cohen's d"  : f"{d1:.1f}",
        "Interpretation": "Significant" if p1 < 0.05 else "Not significant"
    },
    {
        "Test"       : "Paired t-test (rest CV vs. task)",
        "t-statistic": f"t({len(rest_folds)-1}) = {t2:.1f}",
        "p-value"    : f"{p2:.3f}",
        "Cohen's d"  : f"{d2:.2f}",
        "Interpretation": "Significant" if p2 < 0.05 else "Not significant"
    },
])

stats_table.set_index("Test", inplace=True)
stats_table

,t-statistic,p-value,Cohen's d,Interpretation
Test,,,,
One-sample t-test (CV vs. chance),t(4) = 213.7,2.88e-09,95.6,Significant
Paired t-test (rest CV vs. task),t(4) = 11.1,0.000,4.97,Significant


## Cell 10 — Key Finding Summary

In [ ]:
times_above_chance = rest_cv_acc / CHANCE

print("=" * 60)
print("KEY FINDING — H1: Resting State Classification Performance")
print("=" * 60)
print()
print(f"  The full-connectivity one-vs-rest classifier achieved a")
print(f"  resting-state CV accuracy of {rest_cv_acc*100:.2f}%, which is")
print(f"  {times_above_chance:.0f}× above chance level ({CHANCE*100:.2f}%).")
print()
print(f"  Performance was statistically significant vs. chance:")
print(f"    t({len(rest_folds)-1}) = {t1:.1f}, p < 0.001, Cohen's d = {d1:.1f}")
print()
print(f"  Generalisation to held-out task data yielded {task_acc*100:.2f}%,")
print(f"  an absolute drop of {abs_drop*100:.2f}% ({rel_drop:.1f}% relative).")
print(f"  The rest→task drop was significant:")
print(f"    t({len(rest_folds)-1}) = {t2:.1f}, p = {p2:.3f}, Cohen's d = {d2:.2f}")
print()
print(f"  Task error count: {error_count:,} / {n_task_samples:,} samples")
print("=" * 60)

KEY FINDING — H1: Resting State Classification Performance

  The full-connectivity one-vs-rest classifier achieved a
  resting-state CV accuracy of 76.24%, which is
  177× above chance level (0.43%).

  Performance was statistically significant vs. chance:
    t(4) = 213.7, p < 0.001, Cohen's d = 95.6

  Generalisation to held-out task data yielded 72.85%,
  an absolute drop of 3.40% (4.5% relative).
  The rest→task drop was significant:
    t(4) = 11.1, p = 0.000, Cohen's d = 4.97

  Task error count: 12,511 / 46,400 samples
